In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Generate Figures B-F from six model LODO outputs in a single script.

Example:
    python plot_figures_B_to_F_all_in_one.py
    MACHINE2_DIR=/home/t440/machine/machine2 python plot_figures_B_to_F_all_in_one.py
    python plot_figures_B_to_F_all_in_one.py --base-dir /path/to/machine2 --figures B D F

Dependencies: numpy, pandas, matplotlib, seaborn, scikit-learn

Input structure, under --base-dir:
    lr/results/LR_outer_LODO_predictions.csv
    rf/results/RF_outer_LODO_predictions.csv
    svm/results/SVM_outer_LODO_predictions.csv
    gbm/results/GBM_outer_LODO_predictions.csv
    xgboost/results/XGB_outer_LODO_predictions.csv
    catboost/results/CAT_outer_LODO_predictions.csv
    {model}/results/{PREFIX}_feature_stability_summary.csv  (for E/F)

Figure B preferentially uses a donor-level outer CV results CSV in the model
folder (as in the original Figure B); if none exists, it calculates donor-level
AUROC and AP using that model's held-out LODO prediction CSV. Both B's fallback
and Figures C/D select SVM decision_score when available (else probability),
so the same underlying scores are evaluated throughout.

Output directories: Figure_B, Figure_C, Figure_D, Figure_E, Figure_F.
Each figure retains its original plotting implementation and output names.
The original Jupyter-only SVG display calls have been removed: saved SVG files
can be opened directly or displayed by a notebook separately.
"""

import argparse
import glob
import os
import sys
import traceback
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import to_hex, to_rgb
from sklearn.metrics import roc_auc_score, average_precision_score

MODEL_DIRS = {
    "LR": "lr", "RF": "rf", "SVM": "svm", "GBM": "gbm",
    "XGBoost": "xgboost", "CatBoost": "catboost",
}
MODEL_ORDER = list(MODEL_DIRS)
MODEL_COLORS_B = {
    "LR": "#A7ABD1", "RF": "#E1B7DF", "SVM": "#F0F09E",
    "GBM": "#A5D2B8", "XGBoost": "#D89EB4", "CatBoost": "#A4E9E5",
}
PREDICTION_PREFIXES = {
    "LR": "LR", "RF": "RF", "SVM": "SVM", "GBM": "GBM",
    "XGBoost": "XGB", "CatBoost": "CAT",
}


def prediction_file(base_dir, model):
    return (Path(base_dir) / MODEL_DIRS[model] / "results" /
            f"{PREDICTION_PREFIXES[model]}_outer_LODO_predictions.csv")


def donor_metrics_from_predictions(base_dir, model):
    """Recalculate held-out donor metrics (never use fitted/training scores)."""
    path = prediction_file(base_dir, model)
    if not path.is_file():
        raise FileNotFoundError(f"{model}: missing LODO predictions file: {path}")
    df = pd.read_csv(path, dtype={"individual": str})
    df.columns = df.columns.str.strip()
    required = {"individual", "true_label"}
    if not required.issubset(df.columns):
        raise ValueError(f"{model}: prediction file missing {sorted(required - set(df))}: {path}")
    if model == "SVM" and "decision_score" in df:
        score_col = "decision_score"
    elif "predicted_probability" in df:
        score_col = "predicted_probability"
    elif "decision_score" in df:
        score_col = "decision_score"
    else:
        raise ValueError(f"{model}: no continuous score column: {path}")
    df["true_label"] = pd.to_numeric(df["true_label"], errors="raise")
    df[score_col] = pd.to_numeric(df[score_col], errors="raise")
    if df[["individual", "true_label", score_col]].isna().any().any():
        raise ValueError(f"{model}: missing predictions/labels/donor IDs in {path}")
    if not df["true_label"].isin([0, 1]).all():
        raise ValueError(f"{model}: true_label must be 0 or 1 in {path}")
    if not np.isfinite(df[score_col].to_numpy()).all():
        raise ValueError(f"{model}: invalid prediction scores in {path}")
    rows = []
    for donor, sub in df.groupby("individual", sort=True):
        if sub["true_label"].nunique() != 2:
            raise ValueError(f"{model}: donor {donor} has only one label class")
        rows.append({
            "model": model, "donor": str(donor),
            "AUROC": roc_auc_score(sub["true_label"], sub[score_col]),
            "AP": average_precision_score(sub["true_label"], sub[score_col]),
        })
    print(f"[{model}] Calculated per-donor metrics from: {path} ({score_col})")
    return pd.DataFrame(rows)


def find_column(frame, candidates):
    mapping = {str(col).strip().lower(): col for col in frame.columns}
    for candidate in candidates:
        if candidate.lower() in mapping:
            return mapping[candidate.lower()]
    return None


def read_donor_metrics(base_dir, model):
    """Honor Figure B's original donor-level performance files if present."""
    search_dir = Path(base_dir) / MODEL_DIRS[model]
    aliases = {
        "donor": ["test_donor", "outer_test_donor", "held_out_donor",
                  "donor", "individual"],
        "AUROC": ["ROC_AUC", "AUROC", "roc_auc", "auc_roc", "roc_auc_score"],
        "AP": ["AP", "average_precision", "Average_Precision", "auprc",
               "AUPRC", "pr_auc"],
    }
    candidates = []
    for path in glob.glob(str(search_dir / "**" / "*.csv"), recursive=True):
        filename = Path(path).name.lower()
        if any(w in filename for w in ["prediction", "summary", "importance",
                                        "feature", "tuning", "inner", "top20",
                                        "top10", "shap"]):
            continue
        try:
            frame = pd.read_csv(path)
        except (OSError, pd.errors.ParserError, UnicodeError):
            continue
        donor_col = find_column(frame, aliases["donor"])
        auc_col = find_column(frame, aliases["AUROC"])
        ap_col = find_column(frame, aliases["AP"])
        if not all([donor_col, auc_col, ap_col]):
            continue
        score = sum(word in filename for word in ["outer", "lodo", "performance"])
        candidates.append((score, path, frame, donor_col, auc_col, ap_col))

    if not candidates:
        return donor_metrics_from_predictions(base_dir, model)
    candidates.sort(key=lambda x: (-x[0], x[1]))
    top = [c for c in candidates if c[0] == candidates[0][0]]
    if len(top) > 1:
        raise ValueError(
            f"{model}: ambiguous outer-CV files:\n" +
            "\n".join(c[1] for c in top) +
            "\nPlease remove the ambiguous files or choose the correct one."
        )
    _, path, frame, dc, ac, pc = top[0]
    print(f"[{model}] Reading per-donor results: {path}")
    result = pd.DataFrame({
        "model": model, "donor": frame[dc].astype("string").str.strip(),
        "AUROC": pd.to_numeric(frame[ac], errors="coerce"),
        "AP": pd.to_numeric(frame[pc], errors="coerce"),
    })
    if result[["donor", "AUROC", "AP"]].isna().any().any():
        raise ValueError(f"{model}: null/invalid donor-level metrics in {path}")
    if result.duplicated(["model", "donor"]).any():
        raise ValueError(f"{model}: duplicate donor rows in {path}")
    for metric in ["AUROC", "AP"]:
        if not result[metric].between(0, 1).all():
            raise ValueError(f"{model}: invalid {metric} values in {path}")
    return result


def make_figure_b(base_dir):
    """Figure B | donor-level AUROC and AP with original six pastels."""
    out_dir = Path(base_dir) / "Figure_B"
    out_dir.mkdir(parents=True, exist_ok=True)
    results = [read_donor_metrics(base_dir, m) for m in MODEL_ORDER]
    df = pd.concat(results, ignore_index=True)
    df["model"] = pd.Categorical(df["model"], categories=MODEL_ORDER, ordered=True)
    donor_sets = {m: set(df.loc[df["model"] == m, "donor"]) for m in MODEL_ORDER}
    reference = donor_sets[MODEL_ORDER[0]]
    for model, donors in donor_sets.items():
        if donors != reference:
            raise ValueError(
                f"Held-out donors differ for {model}; missing: "
                f"{sorted(reference - donors)}; extra: {sorted(donors - reference)}"
            )
    if len(reference) != 9:
        warnings.warn(f"Figure B: expected 9 AD donors, found {len(reference)}")
    df.to_csv(out_dir / "Figure_B_combined_performance.csv", index=False)
    summary = (df.groupby("model", observed=True)
        .agg(n_donors=("donor", "nunique"),
             AUROC_median=("AUROC", "median"),
             AUROC_Q1=("AUROC", lambda x: x.quantile(0.25)),
             AUROC_Q3=("AUROC", lambda x: x.quantile(0.75)),
             AP_median=("AP", "median"),
             AP_Q1=("AP", lambda x: x.quantile(0.25)),
             AP_Q3=("AP", lambda x: x.quantile(0.75)))
        .reset_index())
    summary.to_csv(out_dir / "Figure_B_summary.csv", index=False)
    print("\nFigure B donor-level summary:\n", summary.to_string(index=False))

    point_colors = {m: to_hex(np.array(to_rgb(c)) * 0.69)
                    for m, c in MODEL_COLORS_B.items()}
    line_color = "#344054"
    sns.set_theme(style="white", context="paper")
    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 9, "axes.linewidth": 0.7,
        "axes.labelsize": 10, "xtick.labelsize": 9, "ytick.labelsize": 9,
        "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    })
    np.random.seed(42)
    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.5), sharey=True)
    for ax, (metric, title) in zip(axes, [
        ("AUROC", "AUROC"), ("AP", "Average precision"),
    ]):
        sns.boxplot(
            data=df, x="model", y=metric, order=MODEL_ORDER,
            hue="model", hue_order=MODEL_ORDER, palette=MODEL_COLORS_B,
            dodge=False, ax=ax, width=0.50, saturation=1,
            showfliers=False,
            boxprops={"edgecolor": line_color, "linewidth": 0.85},
            whiskerprops={"color": line_color, "linewidth": 0.8},
            capprops={"color": line_color, "linewidth": 0.8},
            medianprops={"color": line_color, "linewidth": 1.2},
        )
        sns.stripplot(
            data=df, x="model", y=metric, order=MODEL_ORDER,
            hue="model", hue_order=MODEL_ORDER, palette=point_colors,
            dodge=False, ax=ax, jitter=0.12, size=4.3, alpha=0.85,
            edgecolor="white", linewidth=0.35, zorder=3,
        )
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()
        ax.set_title(title, fontsize=11, fontweight="bold", pad=10)
        ax.set_xlabel("")
        ax.set_ylim(0, 1.02)
        ax.set_yticks(np.arange(0, 1.01, 0.2))
        ax.tick_params(axis="x", rotation=35, length=0)
        ax.tick_params(axis="y", direction="out", width=0.7)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(0.7)
        ax.spines["bottom"].set_linewidth(0.7)
        ax.grid(False)
    axes[0].set_ylabel("Predictive performance")
    axes[1].set_ylabel("")
    fig.text(0.015, 0.97, "B", fontsize=16, fontweight="bold", va="top")
    plt.subplots_adjust(left=0.10, right=0.98, bottom=0.22, top=0.86, wspace=0.30)
    for extension in ["pdf", "svg", "png"]:
        path = out_dir / f"Figure_B_Science_palette.{extension}"
        fig.savefig(path, dpi=600, bbox_inches="tight", facecolor="white")
        print("Saved:", path)
    plt.close(fig)
    print("Figure B completed successfully.")



# ===== Original Figure C (preserved plot settings) =====

def make_figure_c(base_dir):
    """Original Figure C source, scoped to a single invocation."""
    # ============================================================
    # Figure C | ROC and Precision-Recall curves
    # AD microglia: HTS vs LTS
    # Six machine-learning models
    # Outer leave-one-donor-out (LODO) predictions
    #
    # Nature / Science-inspired style
    # Colors matched to Figure B
    #
    # Directly run in Jupyter Notebook
    # No argparse / JSON configuration required
    #
    # Outputs:
    #   Figure_C_ROC_PR.svg
    #   Figure_C_curve_points.csv
    #   Figure_C_plot_metrics.csv
    # ============================================================

    from pathlib import Path
    import warnings

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors

    from sklearn.metrics import (
        roc_curve,
        roc_auc_score,
        precision_recall_curve,
        average_precision_score
    )


    # ============================================================
    # 1. Input and output paths
    # ============================================================

    BASE_DIR = Path(base_dir)

    OUTPUT_DIR = BASE_DIR / "Figure_C"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    MODEL_FILES = {
        "LR": BASE_DIR / "lr/results/LR_outer_LODO_predictions.csv",
        "RF": BASE_DIR / "rf/results/RF_outer_LODO_predictions.csv",
        "SVM": BASE_DIR / "svm/results/SVM_outer_LODO_predictions.csv",
        "GBM": BASE_DIR / "gbm/results/GBM_outer_LODO_predictions.csv",
        "XGBoost": BASE_DIR / "xgboost/results/XGB_outer_LODO_predictions.csv",
        "CatBoost": BASE_DIR / "catboost/results/CAT_outer_LODO_predictions.csv"
    }

    MODEL_ORDER = [
        "LR",
        "RF",
        "SVM",
        "GBM",
        "XGBoost",
        "CatBoost"
    ]


    # ============================================================
    # 2. Figure B colors
    # ============================================================

    # Original Figure B palette

    FIGURE_B_COLORS = {
        "LR": "#A7ABD1",        # Lavender blue
        "RF": "#E1B7DF",        # Light mauve
        "SVM": "#F0F09E",       # Pale yellow
        "GBM": "#A5D2B8",       # Sage green
        "XGBoost": "#D89EB4",   # Dusty rose
        "CatBoost": "#A4E9E5"   # Mint teal
    }

    # Darken each Figure B color for clear vector lines.
    # Set COLOR_DARKEN_FACTOR = 1.0 for the exact original colors.

    COLOR_DARKEN_FACTOR = 0.72

    def darken_color(hex_color, factor=0.72):
        rgb = np.array(mcolors.to_rgb(hex_color))
        return mcolors.to_hex(np.clip(rgb * factor, 0, 1))

    MODEL_COLORS = {
        model: darken_color(color, COLOR_DARKEN_FACTOR)
        for model, color in FIGURE_B_COLORS.items()
    }

    # Additional distinction for grayscale printing
    MODEL_LINESTYLES = {
        "LR": "-",
        "RF": (0, (6, 2)),
        "SVM": (0, (3, 2)),
        "GBM": "-.",
        "XGBoost": "-",
        "CatBoost": (0, (1, 2))
    }

    LINE_WIDTH = 2.0


    # ============================================================
    # 3. Read and validate prediction files
    # ============================================================

    def read_predictions(path, model_name):

        if not path.is_file():
            raise FileNotFoundError(
                f"{model_name}: file not found:\n{path}"
            )

        df = pd.read_csv(path)
        df.columns = df.columns.str.strip()

        required = [
            "individual",
            "true_label"
        ]

        missing = [
            col for col in required
            if col not in df.columns
        ]

        if missing:
            raise ValueError(
                f"{model_name}: missing columns {missing}\n"
                f"Available columns: {list(df.columns)}"
            )

        # SVM can use continuous decision scores directly.
        if model_name == "SVM" and "decision_score" in df.columns:
            score_column = "decision_score"

        elif "predicted_probability" in df.columns:
            score_column = "predicted_probability"

        elif "decision_score" in df.columns:
            score_column = "decision_score"

        else:
            raise ValueError(
                f"{model_name}: no continuous prediction score found.\n"
                f"Available columns: {list(df.columns)}"
            )

        keep_columns = [
            "individual",
            "true_label",
            score_column
        ]

        if "cell" in df.columns:
            keep_columns.append("cell")

        df = df[keep_columns].copy()

        # Rename for unified downstream processing
        df = df.rename(
            columns={score_column: "prediction_score"}
        )

        if df.isna().any().any():
            raise ValueError(
                f"{model_name}: missing values detected."
            )

        # Donor ID
        df["individual"] = df["individual"].astype(str)

        # Cell barcode
        if "cell" in df.columns:
            df["cell"] = df["cell"].astype(str)

            if df.duplicated(["individual", "cell"]).any():
                raise ValueError(
                    f"{model_name}: duplicate donor/cell IDs."
                )

        # Binary true labels
        labels = pd.to_numeric(
            df["true_label"],
            errors="raise"
        )

        if not set(labels.unique()).issubset({0, 1}):
            raise ValueError(
                f"{model_name}: true_label must contain 0 and 1 only."
            )

        df["true_label"] = labels.astype(int)

        if df["true_label"].nunique() != 2:
            raise ValueError(
                f"{model_name}: both binary classes are required."
            )

        # Continuous model score
        scores = pd.to_numeric(
            df["prediction_score"],
            errors="raise"
        )

        if not np.isfinite(scores.to_numpy()).all():
            raise ValueError(
                f"{model_name}: invalid prediction scores."
            )

        # Probabilities must lie within [0, 1].
        # SVM decision scores do not have this restriction.
        if score_column == "predicted_probability":
            if not scores.between(0, 1).all():
                raise ValueError(
                    f"{model_name}: probabilities outside [0, 1]."
                )

        df["prediction_score"] = scores.astype(float)

        print(
            f"{model_name:<9} | "
            f"score = {score_column:<22} | "
            f"cells = {len(df):,} | "
            f"donors = {df['individual'].nunique()}"
        )

        return df, score_column


    # ============================================================
    # 4. Load all six model predictions
    # ============================================================

    ALL_DATA = {}

    print("Loading model predictions:\n")

    for model in MODEL_ORDER:

        df, score_column = read_predictions(
            MODEL_FILES[model],
            model
        )

        ALL_DATA[model] = {
            "data": df,
            "score_column": score_column
        }

    print("\nAll six models loaded successfully.")


    # ============================================================
    # 5. Verify consistent test cohorts
    # ============================================================

    reference_df = ALL_DATA["LR"]["data"]

    reference_counts = (
        reference_df
        .groupby(["individual", "true_label"])
        .size()
        .sort_index()
    )

    for model in MODEL_ORDER[1:]:

        current_df = ALL_DATA[model]["data"]

        current_counts = (
            current_df
            .groupby(["individual", "true_label"])
            .size()
            .sort_index()
        )

        if not reference_counts.equals(current_counts):
            raise ValueError(
                f"{model}: donor/class cell counts differ from LR. "
                "Check whether all models used the same test cohort."
            )

        # Exact cell and label comparison when barcodes exist
        if (
            "cell" in reference_df.columns
            and "cell" in current_df.columns
        ):

            compare_columns = [
                "individual",
                "cell",
                "true_label"
            ]

            ref_check = (
                reference_df[compare_columns]
                .sort_values(["individual", "cell"])
                .reset_index(drop=True)
            )

            current_check = (
                current_df[compare_columns]
                .sort_values(["individual", "cell"])
                .reset_index(drop=True)
            )

            if not ref_check.equals(current_check):
                raise ValueError(
                    f"{model}: cell barcodes or labels differ from LR."
                )

        else:
            warnings.warn(
                f"{model}: exact cell matching skipped because "
                "cell barcodes are unavailable in one file."
            )

    print("Evaluation cohort consistency check passed.")


    # ============================================================
    # 6. Calculate ROC, PR, AUROC and AP
    # ============================================================

    ROC_RESULTS = {}
    PR_RESULTS = {}

    metrics_records = []
    curve_records = []

    print("\nModel performance:\n")

    for model in MODEL_ORDER:

        df = ALL_DATA[model]["data"]

        y_true = df["true_label"].to_numpy()
        y_score = df["prediction_score"].to_numpy()

        # -----------------------------------
        # ROC
        # -----------------------------------

        fpr, tpr, _ = roc_curve(
            y_true,
            y_score,
            pos_label=1
        )

        auc_value = roc_auc_score(
            y_true,
            y_score
        )

        # -----------------------------------
        # Precision-Recall
        # -----------------------------------

        precision, recall, _ = precision_recall_curve(
            y_true,
            y_score,
            pos_label=1
        )

        ap_value = average_precision_score(
            y_true,
            y_score,
            pos_label=1
        )

        prevalence = float(np.mean(y_true))

        ROC_RESULTS[model] = {
            "fpr": fpr,
            "tpr": tpr,
            "auc": auc_value
        }

        PR_RESULTS[model] = {
            "recall": recall,
            "precision": precision,
            "ap": ap_value
        }

        metrics_records.append({
            "model": model,
            "score_column": ALL_DATA[model]["score_column"],
            "pooled_AUROC": auc_value,
            "pooled_AP": ap_value,
            "positive_class_prevalence": prevalence,
            "n_cells": len(df),
            "n_donors": df["individual"].nunique()
        })

        # ROC curve coordinates
        for x, y in zip(fpr, tpr):
            curve_records.append({
                "model": model,
                "curve": "ROC",
                "x": float(x),
                "y": float(y)
            })

        # PR curve coordinates
        for x, y in zip(recall, precision):
            curve_records.append({
                "model": model,
                "curve": "PR",
                "x": float(x),
                "y": float(y)
            })

        print(
            f"{model:<9} | "
            f"AUROC = {auc_value:.3f} | "
            f"AP = {ap_value:.3f}"
        )

        if auc_value < 0.5:
            warnings.warn(
                f"{model}: AUROC < 0.5. "
                "Verify positive-class coding and score direction."
            )


    # ============================================================
    # 7. Export plotting data
    # ============================================================

    metrics_df = pd.DataFrame(metrics_records)
    curve_df = pd.DataFrame(curve_records)

    metrics_file = OUTPUT_DIR / "Figure_C_plot_metrics.csv"
    curve_file = OUTPUT_DIR / "Figure_C_curve_points.csv"

    metrics_df.to_csv(metrics_file, index=False)
    curve_df.to_csv(curve_file, index=False)


    # ============================================================
    # 8. Publication plotting settings
    # ============================================================

    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 9,
        "axes.labelsize": 10,
        "axes.titlesize": 11,
        "axes.linewidth": 0.8,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "xtick.major.width": 0.7,
        "ytick.major.width": 0.7,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
        "legend.fontsize": 8,
        "svg.fonttype": "none",
        "savefig.facecolor": "white"
    })

    fig, (ax_roc, ax_pr) = plt.subplots(
        1,
        2,
        figsize=(10.6, 4.9)
    )

    fig.patch.set_facecolor("white")


    # ============================================================
    # 9. ROC curves
    # ============================================================

    for model in MODEL_ORDER:

        result = ROC_RESULTS[model]

        ax_roc.plot(
            result["fpr"],
            result["tpr"],
            color=MODEL_COLORS[model],
            linestyle=MODEL_LINESTYLES[model],
            linewidth=LINE_WIDTH,
            label=f"{model} ({result['auc']:.3f})",
            solid_capstyle="round"
        )

    # Random classifier baseline
    ax_roc.plot(
        [0, 1],
        [0, 1],
        color="#B6BDC5",
        linestyle="--",
        linewidth=1.0,
        zorder=0
    )

    ax_roc.set_xlim(0, 1)
    ax_roc.set_ylim(0, 1)

    ax_roc.set_xlabel("False positive rate")
    ax_roc.set_ylabel("True positive rate")

    ax_roc.set_title(
        "ROC",
        fontsize=11,
        fontweight="bold",
        pad=12
    )


    # ============================================================
    # 10. Precision-Recall curves
    # ============================================================

    for model in MODEL_ORDER:

        result = PR_RESULTS[model]

        # sklearn returns recall in descending order.
        # Reverse for left-to-right display.
        recall = result["recall"][::-1]
        precision = result["precision"][::-1]

        ax_pr.plot(
            recall,
            precision,
            color=MODEL_COLORS[model],
            linestyle=MODEL_LINESTYLES[model],
            linewidth=LINE_WIDTH,
            label=f"{model} ({result['ap']:.3f})",
            solid_capstyle="round"
        )

    # PR baseline = prevalence of class 1
    prevalences = [
        row["positive_class_prevalence"]
        for row in metrics_records
    ]

    if np.allclose(prevalences, prevalences[0]):

        ax_pr.axhline(
            y=prevalences[0],
            color="#B6BDC5",
            linestyle="--",
            linewidth=1.0,
            zorder=0
        )

    ax_pr.set_xlim(0, 1)
    ax_pr.set_ylim(0, 1)

    ax_pr.set_xlabel("Recall")
    ax_pr.set_ylabel("Precision")

    ax_pr.set_title(
        "Precision–recall",
        fontsize=11,
        fontweight="bold",
        pad=12
    )


    # ============================================================
    # 11. Axis appearance
    # ============================================================

    for ax in [ax_roc, ax_pr]:

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.spines["left"].set_color("#3B4651")
        ax.spines["bottom"].set_color("#3B4651")

        ax.tick_params(
            axis="both",
            direction="out",
            colors="#3B4651"
        )

        ax.set_xticks(np.linspace(0, 1, 6))
        ax.set_yticks(np.linspace(0, 1, 6))

        ax.set_aspect(
            "equal",
            adjustable="box"
        )

        ax.grid(False)


    # ============================================================
    # 12. Legends BELOW each panel
    #     Avoid covering ROC / PR curves
    # ============================================================

    LEGEND_STYLE = {
        "loc": "upper center",

        # Centered below each plotting axis
        "bbox_to_anchor": (0.5, -0.23),

        # Six models in 2 columns x 3 rows
        "ncol": 2,

        "frameon": False,
        "fontsize": 8.2,
        "title_fontsize": 8.8,

        "handlelength": 2.7,
        "handletextpad": 0.55,
        "columnspacing": 1.0,
        "labelspacing": 0.48,

        "borderaxespad": 0.0
    }

    ax_roc.legend(
        title="Pooled AUROC",
        **LEGEND_STYLE
    )

    ax_pr.legend(
        title="Pooled AP",
        **LEGEND_STYLE
    )


    # ============================================================
    # 13. Figure panel label
    # ============================================================

    ax_roc.text(
        -0.16,
        1.10,
        "C",
        transform=ax_roc.transAxes,
        fontsize=15,
        fontweight="bold",
        ha="left",
        va="top",
        clip_on=False
    )


    # ============================================================
    # 14. Adjust spacing
    # ============================================================

    # Extra bottom space is reserved for both legends.

    fig.subplots_adjust(
        left=0.08,
        right=0.98,
        top=0.90,
        bottom=0.34,
        wspace=0.27
    )


    # ============================================================
    # 15. Save SVG ONLY
    # ============================================================

    svg_file = OUTPUT_DIR / "Figure_C_ROC_PR.svg"

    fig.savefig(
        svg_file,
        format="svg",
        bbox_inches="tight",
        pad_inches=0.10,
        facecolor="white"
    )

    plt.close(fig)


    # ============================================================
    # 16. Final output
    # ============================================================

    print("\n" + "=" * 55)
    print("Figure C completed")
    print("=" * 55)

    print("\nOutput files:")
    print(svg_file)
    print(curve_file)
    print(metrics_file)

    print("\nAll six model colors:")
    for model in MODEL_ORDER:
        print(
            f"{model:<9} "
            f"Figure B = {FIGURE_B_COLORS[model]} | "
            f"Curve = {MODEL_COLORS[model]}"
        )

    print(
        "\nNote: ROC and PR show pooled out-of-fold curves. "
        "Cells are pooled across donors, so donor-level "
        "Figure B should remain the primary validation result."
    )



# ===== Original Figure D (preserved plot settings) =====

def make_figure_d(base_dir):
    """Original Figure D source, scoped to a single invocation."""
    # ============================================================
    # Figure D
    # Donor-level AP heatmap + HTS prevalence
    #
    # AP color scale: 0.90 to 1.00
    # Colorbar ticks: every 0.02
    #
    # IMPORTANT:
    # All original colors are preserved.
    #
    # Outputs:
    # 1. Figure_D_AP_heatmap_HTS_prevalence_v4.svg
    # 2. Figure_D_plot_data.csv
    # 3. Figure_D_plot_settings.json
    # ============================================================

    from pathlib import Path
    import json
    import warnings

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    from matplotlib.colors import LinearSegmentedColormap, Normalize
    from matplotlib.patches import Rectangle
    from matplotlib.ticker import FormatStrFormatter

    from sklearn.metrics import average_precision_score


    # ============================================================
    # 1. Input and output paths
    # ============================================================

    BASE_DIR = Path(base_dir)

    OUT_DIR = BASE_DIR / "Figure_D"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    FILES = {
        "LR": BASE_DIR / "lr/results/LR_outer_LODO_predictions.csv",
        "RF": BASE_DIR / "rf/results/RF_outer_LODO_predictions.csv",
        "SVM": BASE_DIR / "svm/results/SVM_outer_LODO_predictions.csv",
        "GBM": BASE_DIR / "gbm/results/GBM_outer_LODO_predictions.csv",
        "XGBoost": BASE_DIR / "xgboost/results/XGB_outer_LODO_predictions.csv",
        "CatBoost": BASE_DIR / "catboost/results/CAT_outer_LODO_predictions.csv"
    }

    MODELS = list(FILES.keys())


    # ============================================================
    # 2. Plot settings
    # ============================================================

    # true_label = 1 must correspond to HTS
    POSITIVE_LABEL = 1

    # "median_AP": lowest median AP at the top
    # "donor_id": D1 to D9
    SORT_DONORS = "median_AP"

    # Use D1-D9 instead of original donor IDs
    ANONYMIZE = True

    # ------------------------------------------------------------
    # UPDATED: AP scale and numerical tick spacing
    # ------------------------------------------------------------

    AP_VMIN = 0.90
    AP_VMAX = 1.00

    AP_TICK_STEP = 0.02

    AP_TICKS = np.round(
        np.arange(
            AP_VMIN,
            AP_VMAX + AP_TICK_STEP / 2,
            AP_TICK_STEP
        ),
        2
    )


    # ============================================================
    # 3. Figure B model colors (UNCHANGED)
    # ============================================================

    FIGURE_B_COLORS = {
        "LR": "#A7ABD1",
        "RF": "#E1B7DF",
        "SVM": "#F0F09E",
        "GBM": "#A5D2B8",
        "XGBoost": "#D89EB4",
        "CatBoost": "#A4E9E5"
    }


    # ============================================================
    # 4. Original heatmap gradient colors (UNCHANGED)
    # ============================================================

    HEAT_COLORS = [
        "#F6FAFB",
        "#DDECEF",
        "#B4D2DA",
        "#7CACBF",
        "#3D8099",
        "#174D66"
    ]

    # Original HTS prevalence color (UNCHANGED)
    BAR_COLOR = "#91A6B2"

    # Continuous gradient using the exact original colors
    cmap = LinearSegmentedColormap.from_list(
        "AP_blue_teal_original",
        HEAT_COLORS,
        N=256
    )

    # Apply the original gradient across 0.90-1.00
    norm = Normalize(
        vmin=AP_VMIN,
        vmax=AP_VMAX,
        clip=True
    )


    # ============================================================
    # 5. Read prediction files
    # ============================================================

    def read_predictions(model, path):

        if not path.is_file():
            raise FileNotFoundError(
                f"{model}: missing input file:\n{path}"
            )

        df = pd.read_csv(
            path,
            dtype={
                "individual": str,
                "cell": str
            }
        )

        df.columns = df.columns.str.strip()

        required = {"individual", "true_label"}

        missing = required - set(df.columns)

        if missing:
            raise ValueError(
                f"{model}: missing columns {sorted(missing)}"
            )

        # SVM decision scores are valid continuous scores
        if model == "SVM" and "decision_score" in df.columns:
            score_col = "decision_score"

        elif "predicted_probability" in df.columns:
            score_col = "predicted_probability"

        elif "decision_score" in df.columns:
            score_col = "decision_score"

        else:
            raise ValueError(
                f"{model}: no continuous prediction score found."
            )

        keep = [
            "individual",
            "true_label",
            score_col
        ]

        if "cell" in df.columns:
            keep.append("cell")

        df = df[keep].copy()

        df.rename(
            columns={score_col: "score"},
            inplace=True
        )

        if df.isna().any().any():
            raise ValueError(
                f"{model}: missing values detected."
            )

        df["individual"] = (
            df["individual"]
            .astype(str)
            .str.strip()
        )

        if df["individual"].eq("").any():
            raise ValueError(
                f"{model}: blank donor IDs."
            )

        df["true_label"] = pd.to_numeric(
            df["true_label"],
            errors="raise"
        )

        if not df["true_label"].isin([0, 1]).all():
            raise ValueError(
                f"{model}: true_label must be 0 or 1."
            )

        df["true_label"] = df["true_label"].astype(int)

        df["score"] = pd.to_numeric(
            df["score"],
            errors="raise"
        ).astype(float)

        if not np.isfinite(df["score"]).all():
            raise ValueError(
                f"{model}: invalid prediction scores."
            )

        if score_col == "predicted_probability":

            if not df["score"].between(0, 1).all():
                raise ValueError(
                    f"{model}: probability outside [0, 1]."
                )

        if "cell" in df.columns:

            df["cell"] = (
                df["cell"]
                .astype(str)
                .str.strip()
            )

            if df.duplicated(
                ["individual", "cell"]
            ).any():

                raise ValueError(
                    f"{model}: duplicated donor/cell IDs."
                )

        donor_classes = (
            df.groupby("individual")["true_label"]
            .nunique()
        )

        if not donor_classes.eq(2).all():
            raise ValueError(
                f"{model}: at least one donor lacks a class."
            )

        print(
            f"{model:<9} | "
            f"{len(df):>7,} cells | "
            f"{df['individual'].nunique()} donors | "
            f"{score_col}"
        )

        return df, score_col


    # ============================================================
    # 6. Load six models
    # ============================================================

    DATA = {}
    SCORE_NAMES = {}

    print("Loading predictions:\n")

    for model in MODELS:

        DATA[model], SCORE_NAMES[model] = read_predictions(
            model,
            FILES[model]
        )

    print("\nAll models loaded successfully.")


    # ============================================================
    # 7. Check cohort consistency
    # ============================================================

    reference = DATA["LR"]

    ref_counts = (
        reference
        .groupby(["individual", "true_label"])
        .size()
        .sort_index()
    )

    for model in MODELS[1:]:

        current = DATA[model]

        current_counts = (
            current
            .groupby(["individual", "true_label"])
            .size()
            .sort_index()
        )

        if not ref_counts.equals(current_counts):
            raise ValueError(
                f"{model}: donor/class counts differ from LR."
            )

        if (
            "cell" in reference.columns
            and "cell" in current.columns
        ):

            keys = [
                "individual",
                "cell",
                "true_label"
            ]

            ref_check = (
                reference[keys]
                .sort_values(["individual", "cell"])
                .reset_index(drop=True)
            )

            current_check = (
                current[keys]
                .sort_values(["individual", "cell"])
                .reset_index(drop=True)
            )

            if not ref_check.equals(current_check):
                raise ValueError(
                    f"{model}: cell IDs or labels differ from LR."
                )

        else:
            warnings.warn(
                f"{model}: exact cell matching skipped "
                "because cell IDs are unavailable."
            )

    print("Cohort consistency check passed.")


    # ============================================================
    # 8. Donor labels
    # ============================================================

    ALL_DONORS = sorted(
        reference["individual"].unique().tolist()
    )

    N_DONORS = len(ALL_DONORS)
    N_MODELS = len(MODELS)

    if N_DONORS != 9:
        warnings.warn(
            f"Expected 9 donors, found {N_DONORS}."
        )

    DONOR_ALIAS = {
        donor: f"D{i + 1}"
        for i, donor in enumerate(ALL_DONORS)
    }


    # ============================================================
    # 9. Calculate donor-level AP and HTS prevalence
    # ============================================================

    records = []

    for model in MODELS:

        df = DATA[model]

        for donor, sub in df.groupby("individual"):

            y_true = sub["true_label"].to_numpy()
            y_score = sub["score"].to_numpy()

            ap = average_precision_score(
                y_true,
                y_score,
                pos_label=POSITIVE_LABEL
            )

            n_cells = len(sub)

            n_hts = int(
                np.sum(y_true == POSITIVE_LABEL)
            )

            n_lts = n_cells - n_hts

            prevalence = n_hts / n_cells

            records.append({
                "test_donor": donor,

                "donor_label": (
                    DONOR_ALIAS[donor]
                    if ANONYMIZE
                    else donor
                ),

                "model": model,

                "AP": float(ap),

                "HTS_prevalence": float(prevalence),

                "n_HTS": n_hts,

                "n_LTS": n_lts,

                "n_cells": n_cells,

                "score_column": SCORE_NAMES[model]
            })

    plot_df = pd.DataFrame(records)

    if len(plot_df) != N_DONORS * N_MODELS:
        raise ValueError(
            "Unexpected donor-model record count."
        )

    prevalence_check = (
        plot_df.groupby("test_donor")["HTS_prevalence"]
        .nunique()
    )

    if not prevalence_check.eq(1).all():
        raise ValueError(
            "HTS prevalence differs across models."
        )


    # ============================================================
    # 10. Donor sorting
    # ============================================================

    if SORT_DONORS == "median_AP":

        donor_median_ap = (
            plot_df.groupby("test_donor")["AP"]
            .median()
        )

        DONOR_ORDER = sorted(
            ALL_DONORS,
            key=lambda d: (
                donor_median_ap[d],
                d
            )
        )

    elif SORT_DONORS == "donor_id":

        DONOR_ORDER = list(ALL_DONORS)

    else:
        raise ValueError(
            "SORT_DONORS must be median_AP or donor_id."
        )

    DONOR_LABELS = [
        DONOR_ALIAS[d] if ANONYMIZE else d
        for d in DONOR_ORDER
    ]


    # ============================================================
    # 11. Prepare AP matrix and prevalence
    # ============================================================

    AP_MATRIX = (
        plot_df.pivot(
            index="test_donor",
            columns="model",
            values="AP"
        )
        .loc[DONOR_ORDER, MODELS]
    )

    PREVALENCE = (
        plot_df
        .drop_duplicates("test_donor")
        .set_index("test_donor")
        .loc[DONOR_ORDER, "HTS_prevalence"]
        .to_numpy()
    )

    AP_VALUES = AP_MATRIX.to_numpy(dtype=float)

    print("\nAP range:")
    print(f"Min AP = {AP_VALUES.min():.4f}")
    print(f"Max AP = {AP_VALUES.max():.4f}")

    n_below = int(
        np.sum(AP_VALUES < AP_VMIN)
    )

    if n_below > 0:
        warnings.warn(
            f"{n_below} AP values fall below {AP_VMIN:.2f}. "
            "They will use the lowest gradient color; "
            "their numerical AP labels remain unchanged."
        )


    # ============================================================
    # 12. Save CSV and plotting settings
    # ============================================================

    plot_df["row_order"] = (
        plot_df["test_donor"].map(
            {d: i + 1 for i, d in enumerate(DONOR_ORDER)}
        )
    )

    plot_df["column_order"] = (
        plot_df["model"].map(
            {m: j + 1 for j, m in enumerate(MODELS)}
        )
    )

    plot_df = (
        plot_df
        .sort_values(["row_order", "column_order"])
        .reset_index(drop=True)
    )

    CSV_FILE = OUT_DIR / "Figure_D_plot_data.csv"

    JSON_FILE = OUT_DIR / "Figure_D_plot_settings.json"

    SVG_FILE = (
        OUT_DIR /
        "Figure_D_AP_heatmap_HTS_prevalence_v4.svg"
    )

    plot_df.to_csv(
        CSV_FILE,
        index=False
    )

    settings = {
        "positive_label": POSITIVE_LABEL,
        "donor_sort": SORT_DONORS,
        "anonymize_donors": ANONYMIZE,
        "donor_order": DONOR_ORDER,
        "donor_labels": DONOR_LABELS,
        "models": MODELS,

        "figure_B_model_colors": FIGURE_B_COLORS,
        "heatmap_colors": HEAT_COLORS,
        "prevalence_bar_color": BAR_COLOR,

        "AP_scale_min": AP_VMIN,
        "AP_scale_max": AP_VMAX,
        "AP_tick_step": AP_TICK_STEP,
        "AP_ticks": AP_TICKS.tolist(),

        "input_files": {
            model: str(FILES[model])
            for model in MODELS
        }
    }

    JSON_FILE.write_text(
        json.dumps(
            settings,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )


    # ============================================================
    # 13. Matplotlib style
    # ============================================================

    plt.close("all")

    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 9,
        "axes.linewidth": 0.7,
        "svg.fonttype": "none",
        "savefig.facecolor": "white"
    })


    # ============================================================
    # 14. Figure layout
    # ============================================================

    fig = plt.figure(
        figsize=(10.0, 6.5),
        facecolor="white"
    )

    gs = fig.add_gridspec(
        3,
        2,

        width_ratios=[6.0, 2.8],
        height_ratios=[0.9, 8.5, 0.7],

        left=0.13,
        right=0.96,

        bottom=0.12,
        top=0.82,

        wspace=0.20,
        hspace=0.12
    )

    ax_header = fig.add_subplot(gs[0, 0])
    ax_heat = fig.add_subplot(gs[1, 0])

    ax_bar_header = fig.add_subplot(gs[0, 1])
    ax_bar = fig.add_subplot(gs[1, 1])

    ax_colorbar = fig.add_subplot(gs[2, 0])


    # ============================================================
    # 15. Model headers - original colors
    # ============================================================

    ax_header.set_xlim(
        -0.5,
        N_MODELS - 0.5
    )

    ax_header.set_ylim(0, 1)

    for j, model in enumerate(MODELS):

        ax_header.text(
            j,
            0.72,
            model,

            ha="center",
            va="center",

            fontsize=9,
            fontweight="semibold",
            color="#253B4B"
        )

        ax_header.add_patch(
            Rectangle(
                (j - 0.44, 0.10),

                width=0.88,
                height=0.21,

                facecolor=FIGURE_B_COLORS[model],
                edgecolor="none"
            )
        )

    ax_header.axis("off")


    # ============================================================
    # 16. Prevalence header
    # ============================================================

    ax_bar_header.text(
        0.5,
        0.70,
        "HTS prevalence",

        ha="center",
        va="center",

        fontsize=10,
        fontweight="semibold",
        color="#253B4B",

        transform=ax_bar_header.transAxes
    )

    ax_bar_header.axis("off")


    # ============================================================
    # 17. AP heatmap - updated color scale only
    # ============================================================

    heatmap_image = ax_heat.imshow(
        AP_VALUES,

        cmap=cmap,
        norm=norm,

        interpolation="nearest",
        aspect="auto"
    )

    ax_heat.set_yticks(
        range(N_DONORS),
        labels=DONOR_LABELS
    )

    ax_heat.set_xticks([])

    ax_heat.set_ylabel(
        "AD donor",
        fontsize=10,
        labelpad=10
    )

    ax_heat.tick_params(
        axis="y",
        length=0,
        pad=8,
        labelsize=10
    )

    # White borders
    ax_heat.set_xticks(
        np.arange(-0.5, N_MODELS, 1),
        minor=True
    )

    ax_heat.set_yticks(
        np.arange(-0.5, N_DONORS, 1),
        minor=True
    )

    ax_heat.grid(
        which="minor",
        color="white",
        linewidth=1.5
    )

    ax_heat.tick_params(
        which="minor",
        length=0
    )

    for spine in ax_heat.spines.values():
        spine.set_visible(False)


    # ============================================================
    # 18. AP numeric labels
    # ============================================================

    for i in range(N_DONORS):

        for j in range(N_MODELS):

            ap = AP_VALUES[i, j]

            r, g, b, _ = cmap(norm(ap))

            brightness = (
                0.2126 * r
                + 0.7152 * g
                + 0.0722 * b
            )

            text_color = (
                "#FFFFFF"
                if brightness < 0.60
                else "#233B49"
            )

            ax_heat.text(
                j,
                i,
                f"{ap:.2f}",

                ha="center",
                va="center",

                fontsize=9.2,
                color=text_color
            )


    # ============================================================
    # 19. HTS prevalence bars
    # ============================================================

    y_positions = np.arange(N_DONORS)

    ax_bar.barh(
        y_positions,
        PREVALENCE * 100,

        height=0.74,

        color=BAR_COLOR,
        edgecolor="white",
        linewidth=0.5
    )

    ax_bar.set_xlim(0, 115)

    ax_bar.set_xticks(
        [0, 25, 50, 75, 100],
        labels=["0", "25", "50", "75", "100"]
    )

    ax_bar.set_yticks([])

    ax_bar.tick_params(
        axis="x",
        labelsize=8,
        length=3,
        colors="#596C78"
    )

    ax_bar.grid(
        axis="x",
        color="#E7EDF0",
        linewidth=0.7
    )

    ax_bar.set_axisbelow(True)

    for i, p in enumerate(PREVALENCE):

        ax_bar.text(
            p * 100 + 2,
            i,
            f"{p:.0%}",

            va="center",
            ha="left",

            fontsize=9,
            color="#304758"
        )

    for spine in ["top", "right", "left"]:
        ax_bar.spines[spine].set_visible(False)

    ax_bar.spines["bottom"].set_color("#BCC7CD")

    # Precisely align rows
    ax_heat.set_ylim(
        N_DONORS - 0.5,
        -0.5
    )

    ax_bar.set_ylim(
        N_DONORS - 0.5,
        -0.5
    )


    # ============================================================
    # 20. UPDATED AP colorbar
    #
    # Range: 0.90–1.00
    # Equal numerical spacing: 0.02
    # Original gradient colors are unchanged.
    # ============================================================

    colorbar = fig.colorbar(
        heatmap_image,

        cax=ax_colorbar,

        orientation="horizontal",

        ticks=AP_TICKS
    )

    # Display exactly two decimal places
    colorbar.ax.xaxis.set_major_formatter(
        FormatStrFormatter("%.2f")
    )

    # Explicitly use the chosen range
    colorbar.ax.set_xlim(
        AP_VMIN,
        AP_VMAX
    )

    colorbar.set_label(
        "Average precision (AP)",
        fontsize=9,
        labelpad=7
    )

    # Give the six numbers more breathing room
    colorbar.ax.tick_params(
        axis="x",
        labelsize=8.5,
        length=3,
        pad=5
    )

    colorbar.outline.set_visible(False)


    # ============================================================
    # 21. Panel label and title
    # ============================================================

    fig.text(
        0.05,
        0.956,
        "D",

        fontsize=16,
        fontweight="bold",

        ha="left",
        va="top",

        color="#233B49"
    )

    fig.text(
        0.13,
        0.955,

        "Donor-level HTS identification",

        fontsize=13,
        fontweight="semibold",

        ha="left",
        va="top",

        color="#233B49"
    )

    fig.text(
        0.13,
        0.902,

        "Average precision across six classifiers, "
        "with donor-specific HTS prevalence",

        fontsize=9,

        ha="left",
        va="top",

        color="#687A85"
    )


    # ============================================================
    # 22. Save SVG
    # ============================================================

    fig.savefig(
        SVG_FILE,

        format="svg",

        bbox_inches="tight",
        pad_inches=0.13,

        facecolor="white"
    )

    plt.close(fig)


    # ============================================================
    # 23. Summary
    # ============================================================

    print("\n" + "=" * 60)
    print("FIGURE D COMPLETED")
    print("=" * 60)

    print("\nSaved files:")
    print("SVG :", SVG_FILE)
    print("CSV :", CSV_FILE)
    print("JSON:", JSON_FILE)

    print("\nAP color scale:")
    print(f"Range: {AP_VMIN:.2f} - {AP_VMAX:.2f}")
    print(f"Tick interval: {AP_TICK_STEP:.2f}")

    print("Ticks:", AP_TICKS.tolist())

    print("\nDonor order (top to bottom):")

    for donor in DONOR_ORDER:
        print(f"{DONOR_ALIAS[donor]} = {donor}")

    print("\nOriginal colors preserved.")





# ===== Original Figure E (preserved plot settings) =====

def make_figure_e(base_dir):
    """Original Figure E source, scoped to a single invocation."""
    # ============================================================
    # Figure E | Overlap of Top 10 feature genes
    #
    # Six ML models:
    # LR, RF, SVM, GBM, XGBoost, CatBoost
    #
    # Automatically detects feature-ranking columns.
    # LR uses mean_abs_coefficient when available.
    #
    # Creates a publication-style UpSet plot using Matplotlib.
    # No upsetplot package required.
    #
    # Outputs:
    # 1. Figure_E_UpSet_top10_overlap_v2.svg
    # 2. Figure_E_top10_gene_lists.csv
    # 3. Figure_E_gene_membership_matrix.csv
    # 4. Figure_E_exclusive_intersections.csv
    # ============================================================

    from pathlib import Path
    from collections import defaultdict
    import warnings

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt


    # ============================================================
    # 1. File paths
    # ============================================================

    BASE_DIR = Path(base_dir)

    OUT_DIR = BASE_DIR / "Figure_E"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    MODEL_FILES = {
        "LR":
            BASE_DIR / "lr/results/LR_feature_stability_summary.csv",

        "RF":
            BASE_DIR / "rf/results/RF_feature_stability_summary.csv",

        "SVM":
            BASE_DIR / "svm/results/SVM_feature_stability_summary.csv",

        "GBM":
            BASE_DIR / "gbm/results/GBM_feature_stability_summary.csv",

        "XGBoost":
            BASE_DIR / "xgboost/results/XGB_feature_stability_summary.csv",

        "CatBoost":
            BASE_DIR / "catboost/results/CAT_feature_stability_summary.csv"
    }

    MODEL_ORDER = list(MODEL_FILES.keys())

    TOP_N = 10


    # ============================================================
    # 2. Model colors (same as Figure B)
    # ============================================================

    MODEL_COLORS = {
        "LR": "#A7ABD1",
        "RF": "#E1B7DF",
        "SVM": "#F0F09E",
        "GBM": "#A5D2B8",
        "XGBoost": "#D89EB4",
        "CatBoost": "#A4E9E5"
    }

    UPSET_COLOR = "#39489F"
    INACTIVE_COLOR = "#D9DFE6"


    # ============================================================
    # 3. Candidate ranking columns
    #
    # True  = smaller value is better (rank)
    # False = larger value is better (importance)
    #
    # IMPORTANT:
    # mean_abs_coefficient is now included for LR.
    # ============================================================

    RANKING_CANDIDATES = [

        # Explicit ranks
        ("final_rank", True),
        ("consensus_rank", True),
        ("median_rank", True),
        ("mean_rank", True),
        ("rank", True),

        # Coefficient magnitude
        ("mean_abs_coefficient", False),
        ("mean_abs_coef", False),
        ("abs_coefficient", False),
        ("abs_coef", False),
        ("coef_abs", False),

        # SHAP importance
        ("mean_abs_shap", False),
        ("mean_abs_shap_value", False),
        ("mean_shap_abs", False),

        # General importance
        ("mean_abs_importance", False),
        ("mean_importance", False),
        ("mean_feature_importance", False),
        ("feature_importance_mean", False),
        ("importance_mean", False),
        ("average_importance", False),
        ("importance", False),
        ("importance_score", False),

        # Permutation importance
        ("mean_permutation_importance", False),
        ("permutation_importance_mean", False),
        ("mean_perm_importance", False),

        # Tree-based importance
        ("mean_gain", False),
        ("total_gain", False),
        ("gain", False),
        ("mean_split_importance", False),

        # Fallback: feature-selection stability
        ("selection_frequency", False),
        ("top20_count", False)
    ]


    # ============================================================
    # 4. Extract Top 10 genes
    # ============================================================

    def extract_top_genes(filepath, model_name, top_n=10):

        if not filepath.is_file():

            raise FileNotFoundError(
                f"{model_name}: file not found:\n{filepath}"
            )

        df = pd.read_csv(filepath)

        df.columns = df.columns.str.strip()

        # Identify gene column
        gene_candidates = [
            "gene",
            "Gene",
            "GENE",
            "feature",
            "Feature",
            "features",
            "gene_symbol",
            "Gene_symbol",
            "gene_name"
        ]

        gene_col = next(
            (
                col for col in gene_candidates
                if col in df.columns
            ),
            None
        )

        if gene_col is None:

            raise ValueError(
                f"{model_name}: gene column not found.\n"
                f"Available columns: {list(df.columns)}"
            )

        # Identify ranking or importance column
        selected_column = None
        ascending = None

        for col, ascending_order in RANKING_CANDIDATES:

            if col in df.columns:

                selected_column = col
                ascending = ascending_order
                break

        if selected_column is None:

            raise ValueError(
                f"{model_name}: no recognized ranking "
                f"or importance column.\n"
                f"Available columns: {list(df.columns)}"
            )

        # Prepare data
        selected_df = pd.DataFrame({

            "gene": df[gene_col],

            "ranking_value": pd.to_numeric(
                df[selected_column],
                errors="coerce"
            )
        })

        selected_df = selected_df.dropna(
            subset=["gene", "ranking_value"]
        ).copy()

        selected_df["gene"] = (
            selected_df["gene"]
            .astype(str)
            .str.strip()
        )

        selected_df = selected_df.loc[
            selected_df["gene"] != ""
        ].copy()

        # Sort genes
        selected_df = selected_df.sort_values(
            by=["ranking_value", "gene"],
            ascending=[ascending, True],
            kind="mergesort"
        )

        # Each gene should occur only once
        selected_df = selected_df.drop_duplicates(
            subset="gene",
            keep="first"
        )

        if len(selected_df) < top_n:

            raise ValueError(
                f"{model_name}: only {len(selected_df)} "
                f"valid unique genes; need {top_n}."
            )

        # Select top N genes
        selected_df = (
            selected_df
            .head(top_n)
            .reset_index(drop=True)
        )

        selected_df["model"] = model_name

        selected_df["rank_in_model"] = (
            np.arange(1, len(selected_df) + 1)
        )

        selected_df["ranking_column"] = selected_column

        # Explicit warning when fallback uses stability
        if selected_column in [
            "selection_frequency",
            "top20_count"
        ]:

            warnings.warn(
                f"{model_name}: using {selected_column} "
                "as a fallback ranking criterion. "
                "This represents selection stability "
                "rather than feature-effect magnitude."
            )

        print(
            f"{model_name:<9} | "
            f"ranking column: {selected_column}"
        )

        print(
            "Top 10:",
            selected_df["gene"].tolist()
        )

        print()

        return selected_df[
            [
                "model",
                "rank_in_model",
                "gene",
                "ranking_column",
                "ranking_value"
            ]
        ]


    # ============================================================
    # 5. Read all six models
    # ============================================================

    print("=" * 60)
    print("EXTRACTING TOP 10 FEATURE GENES")
    print("=" * 60)
    print()

    top10_tables = []
    top10_sets = {}

    for model in MODEL_ORDER:

        result = extract_top_genes(
            MODEL_FILES[model],
            model,
            top_n=TOP_N
        )

        top10_tables.append(result)

        top10_sets[model] = set(
            result["gene"].tolist()
        )

    top10_df = pd.concat(
        top10_tables,
        ignore_index=True
    )

    # Verify 10 unique genes per model
    for model in MODEL_ORDER:

        if len(top10_sets[model]) != TOP_N:

            raise ValueError(
                f"{model}: Top10 gene count is not 10."
            )


    # ============================================================
    # 6. Build gene membership matrix
    # ============================================================

    all_genes = sorted(
        set().union(*top10_sets.values())
    )

    membership_records = []

    for gene in all_genes:

        record = {"gene": gene}

        for model in MODEL_ORDER:

            record[model] = int(
                gene in top10_sets[model]
            )

        record["n_models"] = sum(
            record[model]
            for model in MODEL_ORDER
        )

        membership_records.append(record)

    membership_df = pd.DataFrame(
        membership_records
    )

    membership_df = membership_df.sort_values(
        by=["n_models", "gene"],
        ascending=[False, True]
    ).reset_index(drop=True)


    # ============================================================
    # 7. Calculate EXACT intersections
    #
    # An intersection consists of genes appearing in
    # exactly that combination of models.
    # ============================================================

    intersection_groups = defaultdict(list)

    for gene in all_genes:

        models_containing_gene = tuple(
            model
            for model in MODEL_ORDER
            if gene in top10_sets[model]
        )

        intersection_groups[
            models_containing_gene
        ].append(gene)


    # Order intersections:
    # 1. Larger intersections first
    # 2. More models first when counts tie
    # 3. Fixed model order
    intersections = sorted(
        intersection_groups.items(),

        key=lambda item: (
            -len(item[1]),
            -len(item[0]),
            tuple(
                MODEL_ORDER.index(model)
                for model in item[0]
            )
        )
    )

    intersection_records = []

    for i, (members, genes) in enumerate(
        intersections,
        start=1
    ):

        intersection_records.append({
            "intersection_id": i,

            "models": " & ".join(members),

            "n_models": len(members),

            "intersection_size": len(genes),

            "genes": "; ".join(sorted(genes))
        })

    intersection_df = pd.DataFrame(
        intersection_records
    )


    # ============================================================
    # 8. Print important results
    # ============================================================

    print("=" * 60)
    print("GENE OVERLAP SUMMARY")
    print("=" * 60)

    print(
        f"Models: {len(MODEL_ORDER)}"
    )

    print(
        f"Top genes per model: {TOP_N}"
    )

    print(
        f"Unique genes across all models: {len(all_genes)}"
    )

    print(
        f"Nonempty exclusive intersections: "
        f"{len(intersections)}"
    )

    print("\nGenes shared by all 6 models:")

    shared_six = membership_df.loc[
        membership_df["n_models"] == 6,
        "gene"
    ].tolist()

    print(
        shared_six if shared_six else "None"
    )

    print("\nGenes shared by at least 3 models:")

    shared_three = membership_df.loc[
        membership_df["n_models"] >= 3,
        ["gene", "n_models"]
    ]

    print(
        shared_three.to_string(index=False)
        if not shared_three.empty
        else "None"
    )


    # ============================================================
    # 9. Save data
    # ============================================================

    TOP10_FILE = (
        OUT_DIR / "Figure_E_top10_gene_lists.csv"
    )

    MEMBERSHIP_FILE = (
        OUT_DIR / "Figure_E_gene_membership_matrix.csv"
    )

    INTERSECTION_FILE = (
        OUT_DIR / "Figure_E_exclusive_intersections.csv"
    )

    SVG_FILE = (
        OUT_DIR / "Figure_E_UpSet_top10_overlap_v2.svg"
    )

    top10_df.to_csv(
        TOP10_FILE,
        index=False
    )

    membership_df.to_csv(
        MEMBERSHIP_FILE,
        index=False
    )

    intersection_df.to_csv(
        INTERSECTION_FILE,
        index=False
    )


    # ============================================================
    # 10. Publication plotting style
    # ============================================================

    plt.close("all")

    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 9,
        "axes.linewidth": 0.7,
        "svg.fonttype": "none",
        "savefig.facecolor": "white"
    })

    N_MODELS = len(MODEL_ORDER)
    N_INTERSECTIONS = len(intersections)

    # Expand figure if many intersection combinations exist
    FIG_WIDTH = max(
        11.0,
        5.0 + N_INTERSECTIONS * 0.46
    )

    fig = plt.figure(
        figsize=(FIG_WIDTH, 7.0),
        facecolor="white"
    )

    gs = fig.add_gridspec(
        nrows=2,
        ncols=2,

        width_ratios=[1.45, 7.6],

        height_ratios=[2.65, 2.15],

        left=0.12,
        right=0.97,

        bottom=0.13,
        top=0.82,

        wspace=0.09,
        hspace=0.10
    )

    ax_empty = fig.add_subplot(gs[0, 0])
    ax_bars = fig.add_subplot(gs[0, 1])

    ax_set_sizes = fig.add_subplot(gs[1, 0])
    ax_matrix = fig.add_subplot(gs[1, 1])

    ax_empty.axis("off")


    # ============================================================
    # 11. Top intersection-size bars
    # ============================================================

    x_positions = np.arange(N_INTERSECTIONS)

    intersection_sizes = np.array([
        len(genes)
        for members, genes in intersections
    ])

    max_size = int(
        intersection_sizes.max()
    )

    ax_bars.bar(
        x_positions,
        intersection_sizes,

        width=0.66,

        color=UPSET_COLOR,

        edgecolor="none"
    )

    # Display counts above bars
    for x, count in zip(
        x_positions,
        intersection_sizes
    ):

        ax_bars.text(
            x,
            count + max(0.10, max_size * 0.035),

            str(count),

            ha="center",
            va="bottom",

            fontsize=8.5,
            color="#293E50"
        )

    ax_bars.set_xlim(
        -0.55,
        N_INTERSECTIONS - 0.45
    )

    ax_bars.set_ylim(
        0,
        max_size * 1.30 + 0.2
    )

    ax_bars.set_ylabel(
        "Exclusive intersection size\n(genes)",
        fontsize=10
    )

    ax_bars.set_xticks([])

    ax_bars.spines[
        ["top", "right", "bottom"]
    ].set_visible(False)

    ax_bars.tick_params(
        axis="y",
        labelsize=9,
        length=3
    )


    # ============================================================
    # 12. Bottom UpSet membership matrix
    # ============================================================

    for row_idx in range(N_MODELS):

        # Subtle alternating row background
        if row_idx % 2 == 0:

            ax_matrix.axhspan(
                row_idx - 0.5,
                row_idx + 0.5,

                color="#F5F7FA",

                zorder=0
            )

    for column_idx, (members, genes) in enumerate(
        intersections
    ):

        active_rows = [
            MODEL_ORDER.index(model)
            for model in members
        ]

        # Inactive dots for all six models
        ax_matrix.scatter(
            np.repeat(column_idx, N_MODELS),
            np.arange(N_MODELS),

            s=36,
            color=INACTIVE_COLOR,

            edgecolors="none",
            zorder=1
        )

        # Connect selected models
        if len(active_rows) > 1:

            ax_matrix.plot(
                [column_idx, column_idx],

                [
                    min(active_rows),
                    max(active_rows)
                ],

                color=UPSET_COLOR,

                linewidth=1.6,
                zorder=2
            )

        # Active dots
        ax_matrix.scatter(
            np.repeat(
                column_idx,
                len(active_rows)
            ),

            active_rows,

            s=65,
            color=UPSET_COLOR,

            edgecolors="white",
            linewidths=0.3,

            zorder=3
        )

    ax_matrix.set_xlim(
        -0.55,
        N_INTERSECTIONS - 0.45
    )

    ax_matrix.set_ylim(
        N_MODELS - 0.5,
        -0.5
    )

    ax_matrix.set_xticks([])
    ax_matrix.set_yticks([])

    for spine in ax_matrix.spines.values():
        spine.set_visible(False)


    # ============================================================
    # 13. Left set-size bars
    # ============================================================

    model_y = np.arange(N_MODELS)

    set_sizes = [
        len(top10_sets[model])
        for model in MODEL_ORDER
    ]

    ax_set_sizes.barh(
        model_y,
        set_sizes,

        height=0.65,

        color=[
            MODEL_COLORS[model]
            for model in MODEL_ORDER
        ],

        edgecolor="none"
    )

    ax_set_sizes.set_yticks(
        model_y,
        labels=MODEL_ORDER
    )

    ax_set_sizes.set_ylim(
        N_MODELS - 0.5,
        -0.5
    )

    # Horizontal bars point to the left
    ax_set_sizes.set_xlim(12, 0)

    ax_set_sizes.set_xticks(
        [0, 5, 10]
    )

    ax_set_sizes.tick_params(
        axis="y",
        length=0,
        pad=8,
        labelsize=9.5
    )

    ax_set_sizes.tick_params(
        axis="x",
        labelsize=8
    )

    ax_set_sizes.set_xlabel(
        "Set size",
        fontsize=9
    )

    for spine in ["top", "right", "left"]:
        ax_set_sizes.spines[spine].set_visible(False)


    # ============================================================
    # 14. Figure title and panel label
    # ============================================================

    fig.text(
        0.04,
        0.96,

        "E",

        ha="left",
        va="top",

        fontsize=17,
        fontweight="bold",
        color="#263747"
    )

    fig.text(
        0.12,
        0.953,

        "Overlap of top 10 feature genes across six classifiers",

        ha="left",
        va="top",

        fontsize=13.5,
        fontweight="semibold",
        color="#263747"
    )

    fig.text(
        0.12,
        0.903,

        f"{len(all_genes)} unique genes | "
        f"{N_INTERSECTIONS} nonempty exclusive intersections",

        ha="left",
        va="top",

        fontsize=9.3,
        color="#687A85"
    )


    # ============================================================
    # 15. Save SVG
    # ============================================================

    fig.savefig(
        SVG_FILE,

        format="svg",

        bbox_inches="tight",
        pad_inches=0.12,

        facecolor="white"
    )

    plt.close(fig)


    # ============================================================
    # 16. Output summary
    # ============================================================

    print("\n" + "=" * 60)
    print("FIGURE E COMPLETED")
    print("=" * 60)

    print("\nSVG:")
    print(SVG_FILE)

    print("\nTop10 gene lists:")
    print(TOP10_FILE)

    print("\nGene membership matrix:")
    print(MEMBERSHIP_FILE)

    print("\nExclusive intersections:")
    print(INTERSECTION_FILE)





# ===== Original Figure F (preserved plot settings) =====

def make_figure_f(base_dir):
    """Original Figure F source, scoped to a single invocation."""
    # ============================================================
    # Figure F | Cross-model feature importance rank heatmap
    # FINAL VERSION - Fixed gene labels
    #
    # Six models:
    # LR, RF, SVM, GBM, XGBoost, CatBoost
    #
    # Left: Gene names
    # Center: Feature importance ranks
    # Right: Number of models selecting each gene
    #
    # Rank 1  = darkest blue
    # Rank 10 = lightest blue
    # Gray    = not selected in Top 10
    #
    # Outputs:
    #   Figure_F_feature_rank_heatmap.svg
    #   Figure_F_feature_rank_matrix.csv
    #   Figure_F_feature_rank_long.csv
    #   Figure_F_plot_settings.csv
    #
    # Jupyter Notebook / Python 3.12
    # ============================================================

    from pathlib import Path
    import warnings

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    from matplotlib.colors import (
        LinearSegmentedColormap,
        Normalize
    )
    from matplotlib.patches import Rectangle
    from matplotlib.cm import ScalarMappable


    # ============================================================
    # 1. Paths
    # ============================================================

    BASE_DIR = Path(base_dir)

    OUT_DIR = BASE_DIR / "Figure_F"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    MODEL_FILES = {
        "LR":
            BASE_DIR / "lr/results/LR_feature_stability_summary.csv",

        "RF":
            BASE_DIR / "rf/results/RF_feature_stability_summary.csv",

        "SVM":
            BASE_DIR / "svm/results/SVM_feature_stability_summary.csv",

        "GBM":
            BASE_DIR / "gbm/results/GBM_feature_stability_summary.csv",

        "XGBoost":
            BASE_DIR / "xgboost/results/XGB_feature_stability_summary.csv",

        "CatBoost":
            BASE_DIR / "catboost/results/CAT_feature_stability_summary.csv"
    }

    MODEL_ORDER = list(MODEL_FILES.keys())


    # ============================================================
    # 2. User settings
    # ============================================================

    TOP_N = 10

    # Show genes appearing in at least 2 models' Top 10
    MIN_MODELS = 2

    # None = show all eligible genes
    MAX_GENES = None

    # "consensus" or "alphabetical"
    SORT_GENES = "consensus"

    SHOW_RANK_NUMBERS = True
    SHOW_N_MODELS = True
    SHOW_INLINE = True

    # Gene name font size
    GENE_FONT_SIZE = 10

    # Adjust for unusually long gene names
    GENE_LABEL_WIDTH = 2.2

    # Cell dimensions
    CELL_WIDTH = 0.90
    CELL_HEIGHT = 0.43


    # ============================================================
    # 3. Colors
    # Same blue-teal palette as Figure D
    # ============================================================

    HEAT_COLORS = [
        "#F6FAFB",
        "#DDECEF",
        "#B4D2DA",
        "#7CACBF",
        "#3D8099",
        "#174D66"
    ]

    MISSING_COLOR = "#E8ECF0"

    TEXT_DARK = "#263747"
    TEXT_MUTED = "#657684"
    GRID_COLOR = "#FFFFFF"

    MODEL_COLORS = {
        "LR": "#A7ABD1",
        "RF": "#E1B7DF",
        "SVM": "#F0F09E",
        "GBM": "#A5D2B8",
        "XGBoost": "#D89EB4",
        "CatBoost": "#A4E9E5"
    }

    BAR_COLOR = "#91A6B2"


    # ============================================================
    # 4. Candidate ranking columns
    #
    # True  = smaller value is better
    # False = larger value is better
    #
    # Same ranking priority as Figure E
    # ============================================================

    RANKING_CANDIDATES = [

        ("final_rank", True),
        ("consensus_rank", True),
        ("median_rank", True),
        ("mean_rank", True),
        ("rank", True),

        ("mean_abs_coefficient", False),
        ("mean_abs_coef", False),
        ("abs_coefficient", False),
        ("abs_coef", False),
        ("coef_abs", False),

        ("mean_abs_shap", False),
        ("mean_abs_shap_value", False),
        ("mean_shap_abs", False),

        ("mean_abs_importance", False),
        ("mean_importance", False),
        ("mean_feature_importance", False),
        ("feature_importance_mean", False),
        ("importance_mean", False),
        ("average_importance", False),
        ("importance", False),
        ("importance_score", False),

        ("mean_permutation_importance", False),
        ("permutation_importance_mean", False),
        ("mean_perm_importance", False),

        ("mean_gain", False),
        ("total_gain", False),
        ("gain", False),
        ("mean_split_importance", False),

        ("selection_frequency", False),
        ("top20_count", False)
    ]


    # ============================================================
    # 5. Extract ranked Top 10 genes
    # ============================================================

    def extract_ranked_features(
        file_path,
        model,
        top_n=10
    ):

        if not file_path.is_file():

            raise FileNotFoundError(
                f"{model}: File not found:\n{file_path}"
            )

        df = pd.read_csv(file_path)
        df.columns = df.columns.str.strip()

        gene_candidates = [
            "gene",
            "Gene",
            "GENE",
            "feature",
            "Feature",
            "features",
            "gene_symbol",
            "Gene_symbol",
            "gene_name"
        ]

        gene_col = next(
            (
                c for c in gene_candidates
                if c in df.columns
            ),
            None
        )

        if gene_col is None:

            raise ValueError(
                f"{model}: Gene column not found.\n"
                f"Available columns: {list(df.columns)}"
            )

        ranking_col = None
        ascending = None

        for col, ascending_value in RANKING_CANDIDATES:

            if col in df.columns:

                ranking_col = col
                ascending = ascending_value
                break

        if ranking_col is None:

            raise ValueError(
                f"{model}: No recognized ranking column.\n"
                f"Available columns: {list(df.columns)}"
            )

        work = pd.DataFrame({
            "gene": df[gene_col],
            "ranking_value": pd.to_numeric(
                df[ranking_col],
                errors="coerce"
            )
        })

        work = work.dropna(
            subset=["gene", "ranking_value"]
        ).copy()

        work["gene"] = (
            work["gene"]
            .astype(str)
            .str.strip()
        )

        work = work.loc[
            work["gene"] != ""
        ].copy()

        work = work.sort_values(
            by=["ranking_value", "gene"],
            ascending=[ascending, True],
            kind="mergesort"
        )

        work = work.drop_duplicates(
            subset="gene",
            keep="first"
        )

        if len(work) < top_n:

            raise ValueError(
                f"{model}: Only {len(work)} "
                f"valid unique genes; need {top_n}."
            )

        work = work.head(top_n).copy()

        work["rank"] = np.arange(
            1,
            top_n + 1
        )

        work["model"] = model
        work["ranking_column"] = ranking_col

        if ranking_col in [
            "selection_frequency",
            "top20_count"
        ]:

            warnings.warn(
                f"{model}: Using {ranking_col} "
                "as a fallback ranking criterion. "
                "This measures selection stability, "
                "not feature-effect magnitude."
            )

        print(
            f"{model:<9} | "
            f"Ranking column: {ranking_col}"
        )

        print(
            "Top 10:",
            work["gene"].tolist()
        )

        print()

        return work[
            [
                "gene",
                "model",
                "rank",
                "ranking_value",
                "ranking_column"
            ]
        ]


    # ============================================================
    # 6. Read six models
    # ============================================================

    print("=" * 65)
    print("FIGURE F | CROSS-MODEL FEATURE RANKING")
    print("=" * 65)
    print()

    all_rank_tables = []

    for model in MODEL_ORDER:

        result = extract_ranked_features(
            MODEL_FILES[model],
            model,
            TOP_N
        )

        all_rank_tables.append(result)

    rank_long = pd.concat(
        all_rank_tables,
        ignore_index=True
    )

    for model in MODEL_ORDER:

        subset = rank_long.loc[
            rank_long["model"] == model
        ]

        if len(subset) != TOP_N:

            raise ValueError(
                f"{model}: Expected {TOP_N} genes, "
                f"found {len(subset)}."
            )

        if subset["gene"].nunique() != TOP_N:

            raise ValueError(
                f"{model}: Duplicate genes detected."
            )


    # ============================================================
    # 7. Gene x model rank matrix
    # ============================================================

    rank_matrix = rank_long.pivot(
        index="gene",
        columns="model",
        values="rank"
    )

    rank_matrix = rank_matrix.reindex(
        columns=MODEL_ORDER
    )

    n_models = (
        rank_matrix
        .notna()
        .sum(axis=1)
        .astype(int)
    )

    mean_rank = rank_matrix.mean(
        axis=1,
        skipna=True
    )

    selected_genes = n_models.loc[
        n_models >= MIN_MODELS
    ].index.tolist()

    if len(selected_genes) == 0:

        raise ValueError(
            f"No genes selected by at least "
            f"{MIN_MODELS} models."
        )

    summary = pd.DataFrame({
        "gene": selected_genes,
        "n_models": n_models.loc[
            selected_genes
        ].values,
        "mean_rank": mean_rank.loc[
            selected_genes
        ].values
    })

    if SORT_GENES == "consensus":

        summary = summary.sort_values(
            by=[
                "n_models",
                "mean_rank",
                "gene"
            ],
            ascending=[
                False,
                True,
                True
            ]
        )

    elif SORT_GENES == "alphabetical":

        summary = summary.sort_values(
            by="gene"
        )

    else:

        raise ValueError(
            "SORT_GENES must be "
            "'consensus' or 'alphabetical'."
        )

    if MAX_GENES is not None:

        summary = summary.head(
            MAX_GENES
        )

    gene_order = summary["gene"].tolist()

    plot_rank_matrix = rank_matrix.loc[
        gene_order,
        MODEL_ORDER
    ]

    N_GENES = len(gene_order)
    N_MODELS = len(MODEL_ORDER)


    # ============================================================
    # 8. Export data
    # ============================================================

    export_matrix = plot_rank_matrix.copy()

    export_matrix.insert(
        0,
        "n_models",
        n_models.loc[
            gene_order
        ].astype(int)
    )

    export_matrix.insert(
        1,
        "mean_rank_among_selected_models",
        mean_rank.loc[
            gene_order
        ].round(3)
    )

    MATRIX_FILE = (
        OUT_DIR / "Figure_F_feature_rank_matrix.csv"
    )

    LONG_FILE = (
        OUT_DIR / "Figure_F_feature_rank_long.csv"
    )

    SETTINGS_FILE = (
        OUT_DIR / "Figure_F_plot_settings.csv"
    )

    SVG_FILE = (
        OUT_DIR / "Figure_F_feature_rank_heatmap.svg"
    )

    export_matrix.to_csv(
        MATRIX_FILE,
        index_label="gene"
    )

    rank_long.to_csv(
        LONG_FILE,
        index=False
    )

    settings = {
        "TOP_N": TOP_N,
        "MIN_MODELS": MIN_MODELS,
        "MAX_GENES": str(MAX_GENES),
        "SORT_GENES": SORT_GENES,
        "SHOW_RANK_NUMBERS": SHOW_RANK_NUMBERS,
        "SHOW_N_MODELS": SHOW_N_MODELS,
        "GENE_FONT_SIZE": GENE_FONT_SIZE
    }

    pd.DataFrame(
        list(settings.items()),
        columns=["setting", "value"]
    ).to_csv(
        SETTINGS_FILE,
        index=False
    )


    # ============================================================
    # 9. Prepare heatmap values
    #
    # Rank 1  -> 10 (dark)
    # Rank 10 -> 1  (light)
    # ============================================================

    rank_values = plot_rank_matrix.to_numpy(
        dtype=float
    )

    heat_values = (
        TOP_N + 1 - rank_values
    )

    masked_values = np.ma.masked_invalid(
        heat_values
    )

    cmap = LinearSegmentedColormap.from_list(
        "Feature_rank_blue_teal",
        HEAT_COLORS
    )

    cmap.set_bad(MISSING_COLOR)

    norm = Normalize(
        vmin=1,
        vmax=TOP_N
    )


    # ============================================================
    # 10. Figure dimensions
    #
    # Use a dedicated axis for gene labels.
    # This avoids clipped or hidden y tick labels.
    # ============================================================

    plt.close("all")

    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 9,
        "axes.linewidth": 0.7,
        "svg.fonttype": "none",
        "savefig.facecolor": "white"
    })

    HEAT_WIDTH = CELL_WIDTH * N_MODELS
    HEAT_HEIGHT = CELL_HEIGHT * N_GENES

    COUNT_WIDTH = 1.6 if SHOW_N_MODELS else 0.0

    FIG_WIDTH = (
        1.0
        + GENE_LABEL_WIDTH
        + HEAT_WIDTH
        + COUNT_WIDTH
        + 0.8
    )

    FIG_HEIGHT = max(
        4.5,
        2.3 + HEAT_HEIGHT
    )

    fig = plt.figure(
        figsize=(FIG_WIDTH, FIG_HEIGHT),
        facecolor="white"
    )

    # Convert physical inches to figure coordinates
    LEFT_MARGIN_IN = 0.50
    BOTTOM_IN = 1.25

    LABEL_GAP_IN = 0.12
    COUNT_GAP_IN = 0.20

    HEAT_LEFT_IN = (
        LEFT_MARGIN_IN
        + GENE_LABEL_WIDTH
        + LABEL_GAP_IN
    )

    HEAT_BOTTOM_IN = BOTTOM_IN

    # Main heatmap
    ax_heat = fig.add_axes([
        HEAT_LEFT_IN / FIG_WIDTH,
        HEAT_BOTTOM_IN / FIG_HEIGHT,
        HEAT_WIDTH / FIG_WIDTH,
        HEAT_HEIGHT / FIG_HEIGHT
    ])

    # Dedicated gene-label axis
    ax_genes = fig.add_axes([
        LEFT_MARGIN_IN / FIG_WIDTH,
        HEAT_BOTTOM_IN / FIG_HEIGHT,
        GENE_LABEL_WIDTH / FIG_WIDTH,
        HEAT_HEIGHT / FIG_HEIGHT
    ])

    # Model color-strip axis
    HEADER_BOTTOM_IN = (
        HEAT_BOTTOM_IN
        + HEAT_HEIGHT
        + 0.10
    )

    ax_header = fig.add_axes([
        HEAT_LEFT_IN / FIG_WIDTH,
        HEADER_BOTTOM_IN / FIG_HEIGHT,
        HEAT_WIDTH / FIG_WIDTH,
        0.22 / FIG_HEIGHT
    ])

    if SHOW_N_MODELS:

        COUNT_LEFT_IN = (
            HEAT_LEFT_IN
            + HEAT_WIDTH
            + COUNT_GAP_IN
        )

        ax_count = fig.add_axes([
            COUNT_LEFT_IN / FIG_WIDTH,
            HEAT_BOTTOM_IN / FIG_HEIGHT,
            COUNT_WIDTH / FIG_WIDTH,
            HEAT_HEIGHT / FIG_HEIGHT
        ])

    else:
        ax_count = None


    # ============================================================
    # 11. Main heatmap
    # ============================================================

    im = ax_heat.imshow(
        masked_values,
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
        aspect="auto"
    )

    ax_heat.set_xlim(
        -0.5,
        N_MODELS - 0.5
    )

    ax_heat.set_ylim(
        N_GENES - 0.5,
        -0.5
    )

    ax_heat.set_xticks(
        np.arange(N_MODELS)
    )

    ax_heat.set_xticklabels(
        MODEL_ORDER,
        fontsize=10,
        fontweight="medium",
        color=TEXT_DARK
    )

    ax_heat.xaxis.tick_top()

    ax_heat.tick_params(
        axis="x",
        length=0,
        pad=9
    )

    # Gene names are drawn on a separate axis
    ax_heat.set_yticks([])

    # Cell borders
    ax_heat.set_xticks(
        np.arange(-0.5, N_MODELS, 1),
        minor=True
    )

    ax_heat.set_yticks(
        np.arange(-0.5, N_GENES, 1),
        minor=True
    )

    ax_heat.grid(
        which="minor",
        color=GRID_COLOR,
        linewidth=1.7
    )

    ax_heat.tick_params(
        which="minor",
        bottom=False,
        left=False
    )

    for spine in ax_heat.spines.values():
        spine.set_visible(False)


    # ============================================================
    # 12. Numeric ranks inside heatmap
    # ============================================================

    if SHOW_RANK_NUMBERS:

        for i in range(N_GENES):

            for j in range(N_MODELS):

                rank = rank_values[i, j]

                if np.isnan(rank):

                    label = "–"
                    label_color = "#9AA7B1"

                else:

                    label = str(int(rank))

                    label_color = (
                        "white"
                        if rank <= 4
                        else TEXT_DARK
                    )

                ax_heat.text(
                    j,
                    i,
                    label,
                    ha="center",
                    va="center",
                    fontsize=9,
                    fontweight="medium",
                    color=label_color
                )


    # ============================================================
    # 13. FIX: Dedicated gene names on the LEFT
    #
    # The gene names are explicit text objects,
    # so they cannot disappear due to shared axes
    # or tick-label visibility settings.
    # ============================================================

    ax_genes.set_xlim(0, 1)

    ax_genes.set_ylim(
        N_GENES - 0.5,
        -0.5
    )

    ax_genes.axis("off")

    for i, gene in enumerate(gene_order):

        ax_genes.text(
            0.98,
            i,
            str(gene),

            ha="right",
            va="center",

            fontsize=GENE_FONT_SIZE,
            fontweight="medium",

            color=TEXT_DARK,

            clip_on=False
        )

    # Gene column heading
    fig.text(
        (
            LEFT_MARGIN_IN
            + GENE_LABEL_WIDTH / 2
        ) / FIG_WIDTH,

        (
            HEADER_BOTTOM_IN
            + 0.11
        ) / FIG_HEIGHT,

        "Gene",

        ha="center",
        va="center",

        fontsize=9,
        fontweight="semibold",
        color=TEXT_DARK
    )


    # ============================================================
    # 14. Model color strips
    # ============================================================

    ax_header.set_xlim(
        -0.5,
        N_MODELS - 0.5
    )

    ax_header.set_ylim(0, 1)

    for j, model in enumerate(MODEL_ORDER):

        ax_header.add_patch(
            Rectangle(
                (j - 0.46, 0.15),
                width=0.92,
                height=0.70,
                facecolor=MODEL_COLORS[model],
                edgecolor="none"
            )
        )

    ax_header.axis("off")


    # ============================================================
    # 15. Right: number of models selecting each gene
    # ============================================================

    if SHOW_N_MODELS:

        counts = n_models.loc[
            gene_order
        ].to_numpy(dtype=int)

        ax_count.barh(
            np.arange(N_GENES),
            counts,

            height=0.64,

            color=BAR_COLOR,
            edgecolor="none"
        )

        ax_count.set_xlim(0, 7.2)

        ax_count.set_ylim(
            N_GENES - 0.5,
            -0.5
        )

        ax_count.set_yticks([])

        ax_count.set_xticks(
            [0, 3, 6]
        )

        ax_count.tick_params(
            axis="x",
            labelsize=8,
            length=2
        )

        for i, count in enumerate(counts):

            ax_count.text(
                count + 0.12,
                i,
                str(count),

                ha="left",
                va="center",

                fontsize=8.5,
                color=TEXT_DARK
            )

        for spine in [
            "top",
            "right",
            "left"
        ]:
            ax_count.spines[spine].set_visible(
                False
            )

        ax_count.set_xlabel(
            "Number of models",
            fontsize=9,
            labelpad=5
        )

        fig.text(
            (
                COUNT_LEFT_IN
                + COUNT_WIDTH / 2
            ) / FIG_WIDTH,

            (
                HEADER_BOTTOM_IN
                + 0.11
            ) / FIG_HEIGHT,

            "Top 10 frequency",

            ha="center",
            va="center",

            fontsize=9,
            fontweight="semibold",
            color=TEXT_DARK
        )


    # ============================================================
    # 16. Title
    # ============================================================

    fig.text(
        0.045,
        0.970,

        "F",

        ha="left",
        va="top",

        fontsize=17,
        fontweight="bold",
        color=TEXT_DARK
    )

    fig.text(
        HEAT_LEFT_IN / FIG_WIDTH,
        0.965,

        "Cross-model ranking of HTS/LTS feature genes",

        ha="left",
        va="top",

        fontsize=13.5,
        fontweight="semibold",
        color=TEXT_DARK
    )

    fig.text(
        HEAT_LEFT_IN / FIG_WIDTH,
        0.925,

        (
            f"Top {TOP_N} genes per classifier | "
            f"Genes selected by at least "
            f"{MIN_MODELS} models"
        ),

        ha="left",
        va="top",

        fontsize=9.2,
        color=TEXT_MUTED
    )


    # ============================================================
    # 17. Colorbar
    #
    # Dark blue = rank 1
    # Light blue = rank 10
    # ============================================================

    CBAR_WIDTH_IN = 3.8
    CBAR_HEIGHT_IN = 0.16

    CBAR_LEFT_IN = (
        HEAT_LEFT_IN
        + (HEAT_WIDTH - CBAR_WIDTH_IN) / 2
    )

    CBAR_BOTTOM_IN = 0.43

    cax = fig.add_axes([
        CBAR_LEFT_IN / FIG_WIDTH,
        CBAR_BOTTOM_IN / FIG_HEIGHT,
        CBAR_WIDTH_IN / FIG_WIDTH,
        CBAR_HEIGHT_IN / FIG_HEIGHT
    ])

    cbar = fig.colorbar(
        im,
        cax=cax,
        orientation="horizontal"
    )

    actual_rank_ticks = [
        1, 3, 5, 7, 10
    ]

    tick_positions = [
        TOP_N + 1 - r
        for r in actual_rank_ticks
    ]

    cbar.set_ticks(tick_positions)

    cbar.set_ticklabels(
        [
            str(r)
            for r in actual_rank_ticks
        ]
    )

    cbar.ax.tick_params(
        labelsize=8,
        length=2,
        pad=4
    )

    cbar.outline.set_visible(False)

    fig.text(
        (
            CBAR_LEFT_IN
            + CBAR_WIDTH_IN / 2
        ) / FIG_WIDTH,

        (
            CBAR_BOTTOM_IN
            + 0.44
        ) / FIG_HEIGHT,

        "Feature importance rank (1 = highest)",

        ha="center",
        va="center",

        fontsize=9,
        color=TEXT_DARK
    )

    fig.text(
        (
            CBAR_LEFT_IN
            + CBAR_WIDTH_IN
            + 0.25
        ) / FIG_WIDTH,

        (
            CBAR_BOTTOM_IN
            + 0.08
        ) / FIG_HEIGHT,

        "Gray = not in Top 10",

        ha="left",
        va="center",

        fontsize=8.5,
        color=TEXT_MUTED
    )


    # ============================================================
    # 18. Save SVG
    # ============================================================

    fig.savefig(
        SVG_FILE,

        format="svg",

        bbox_inches="tight",
        pad_inches=0.20,

        facecolor="white"
    )

    plt.close(fig)


    # ============================================================
    # 19. Summary
    # ============================================================

    print("\n" + "=" * 65)
    print("FIGURE F COMPLETED")
    print("=" * 65)

    print(
        "Unique genes across all models:",
        rank_matrix.shape[0]
    )

    print(
        "Genes displayed:",
        N_GENES
    )

    print("\nDisplayed genes:")

    print(
        summary.to_string(index=False)
    )

    print("\nFiles saved:")

    print("SVG:", SVG_FILE)
    print("Rank matrix:", MATRIX_FILE)
    print("Long table:", LONG_FILE)
    print("Settings:", SETTINGS_FILE)





FIGURE_FUNCTIONS = {
    "B": make_figure_b, "C": make_figure_c, "D": make_figure_d,
    "E": make_figure_e, "F": make_figure_f,
}


def main(argv=None):
    parser = argparse.ArgumentParser(
        description="Create Figures B, C, D, E and F using the six LODO models."
    )
    parser.add_argument(
        "--base-dir", default=os.environ.get("MACHINE2_DIR", "~/machine/machine2"),
        help="Root folder containing lr/, rf/, svm/, gbm/, xgboost/, catboost/."
    )
    parser.add_argument(
        "--figures", nargs="+", choices=list(FIGURE_FUNCTIONS),
        default=list(FIGURE_FUNCTIONS),
        help="One or more figures to create (default: B C D E F)."
    )
    args = parser.parse_args(argv)
    base_dir = Path(args.base_dir).expanduser().resolve()
    if not base_dir.is_dir():
        parser.error(f"Base directory not found: {base_dir}")

    print("=" * 76)
    print("Generating Figures B–F | root:", base_dir)
    print("Figures:", ", ".join(args.figures))
    print("=" * 76)
    failures = {}
    for name in dict.fromkeys(args.figures):
        print("\n" + "=" * 76)
        print(f"Figure {name}")
        print("=" * 76)
        try:
            FIGURE_FUNCTIONS[name](base_dir)
        except Exception as exc:
            failures[name] = str(exc)
            print(f"[FAILED] Figure {name}: {exc}", file=sys.stderr)
            traceback.print_exc()
        finally:
            plt.close("all")

    print("\n" + "=" * 76)
    if failures:
        print("Finished with errors:")
        for name, reason in failures.items():
            print(f"  Figure {name}: {reason}")
        print("Successful figures (if any) are already saved.")
        return 1
    print("All requested figures completed successfully.")
    for name in dict.fromkeys(args.figures):
        print(f"  Figure {name} -> {base_dir / ('Figure_' + name)}")
    return 0


if __name__ == "__main__":
    sys.exit(main())
